*Notebook 06 in `notebooks/arcs/`. Parameters (pre-registered): `analysis/arcs/config.yaml`; decisions and run notes: `analysis/arcs/LOG.md`. The static page `site/arcs/index.html` is built from the same saved outputs.*

In [ ]:
import sys
from pathlib import Path

REPO = Path.cwd().resolve().parents[1]   # notebooks/arcs -> repo root
sys.path.insert(0, str(REPO))

import pandas as pd
import plotly.io as pio
from IPython.display import Markdown, display

from analysis.arcs import s06_shuffle as step

pio.renderers.default = "notebook_connected"
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 120)

## Step 6 · Compared to what (1): is there order structure at all?

**What we do.** An arc claims that the *order* of scenes matters: the story builds, turns and resolves. If
that were false, a script's scenes put in random order would give arcs just like the real one. So for every
script we shuffle the order of its scenes 1,000 times. We recompute D1 and D2 for each shuffle and ask where
the real value falls in that script's own shuffle distribution. We report a z score and an empirical p:
- **D1, one-sided.** In a real story neighbouring scenes tend to share a mood, so smoothing should keep more
  swing than in a random order, where neighbours cancel out.
- **D2, two-sided.** Ending minus start could go either way.

Then we count how many scripts beat their own null. With no order structure about 5% would, by chance.

**What this null does and does not test.** It compares each script with itself: the same scenes, the same
mean tone, only the order differs. It does **not** remove differences *between* scripts, such as genre,
length or budget. That is Step 7's job. Shuffling never changes a script's mean level, so the mean level
can't be tested here.

In [ ]:
res = step.run()   # recomputes this step and saves its outputs

/Users/gyula/Documents/Projects/code/IMDb-movie-success-arcs/analysis/arcs/s06_shuffle.py:58: RuntimeWarning: invalid value encountered in scalar divide
  row |= {k: r[k], f"{k}_null_mean": mu, f"{k}_null_sd": sd, f"{k}_z": (r[k] - mu) / sd}


**Table.** How many scripts beat their own shuffle null, primary sample first. The binomial p tests that share against the 5% expected by chance.

In [ ]:
display(step.tables(res)["aggregate"])

,sample,descriptor,scripts,beyond null (p < 0.05),share,expected share under no order,binomial p (share vs 5%),median z,share z > 0,real / null mean (median)
0,primary 834,D1,834,223,26.7%,5.0%,1.3e-95,0.959,79.9%,1.179
1,primary 834,D2,834,102,12.2%,5.0%,2.3e-16,0.122,52.9%,NaN
2,"all 1,087",D1,1087,281,25.9%,5.0%,5.7e-116,0.936,78.2%,1.169
3,"all 1,087",D2,1087,131,12.1%,5.0%,5.6e-20,0.097,51.7%,NaN


**Figure.** z of each primary script's real D1 and D2 within its own 1,000 shuffles. Dashed: what the z values would look like if scene order carried nothing.

In [ ]:
step.figures(res)["z"].show()

**Figure.** Per-script p-values. Without order structure they would be flat (dashed line).

In [ ]:
step.figures(res)["p"].show()

**Figure.** The first of the Step 5 example scripts: its 1,000 shuffled D1 values and the real one.

In [ ]:
step.figures(res)["example"].show()

**Table.** Order structure by genre bin (primary sample).

In [ ]:
display(step.tables(res)["genres"])

,primary_genre,films,median_D1_z,D1_beyond,median_D2_z
0,Action,66,0.78,20%,0.25
1,Adventure,113,1.21,28%,0.14
2,Comedy,175,0.83,23%,0.22
3,Drama,203,0.98,33%,-0.02
4,Horror,98,1.01,33%,-0.12
5,Other,4,1.00,25%,-0.62
6,Sci-Fi,71,0.93,25%,0.21
7,Thriller,104,1.04,18%,0.19


**Table.** Scripts flagged as possible transcripts against the rest (all scripts). A transcript has little description, so its arc rests on dialogue alone.

In [ ]:
display(step.tables(res)["transcripts"])

,possible_transcript,scripts,median_D1_z,D1_beyond
0,False,1032,0.96,27%
1,True,55,0.35,5%


**Plain-language reading** (generated from the saved numbers):

In [ ]:
display(Markdown(step.reading(res)))

Shuffling each script's scenes 1,000 times: the real volatility (D1) lies beyond its own shuffle null (one-sided p < 0.05) for 223 of 834 primary scripts (27%, against 5% expected if order carried nothing; binomial p 1.3e-95). The median z is +0.96, 80% of scripts have z > 0, and the real D1 is typically 1.18 times the shuffled mean. So the order of scenes does carry structure: neighbouring scenes share tone more than a random order would, and the smoothed arc keeps that. For D2 (two-sided), 102 scripts (12%) lie beyond the null (binomial p 2.3e-16). The median z is +0.12 and only 53% of scripts have z > 0. So endings differ from a random placement of the same scenes more often than chance allows. This is a within-script test. It says nothing yet about differences between films, or about profit, and it rests on DistilBERT SST-2 scores that are NOT YET VALIDATED.